In [2]:
print("hello world")

hello world


In [3]:
import json
import pandas as pd
from openai import OpenAI

In [4]:
# ---------------------------------------
# 1. Initialize OpenAI client
# ---------------------------------------

client = OpenAI()


In [6]:
# ---------------------------------------
# 2. Load CSV data
# ---------------------------------------

df = pd.read_csv("transcriptions.csv")

print("Number of records:", len(df))
print(df.head())

Number of records: 54
  medical_specialty                                      transcription
0        Cardiology  A 65-year-old male presents with chest pain on...
1        Cardiology  A 58-year-old woman reports intermittent palpi...
2        Cardiology  A 72-year-old male has shortness of breath and...
3        Cardiology  A 49-year-old patient has elevated cholesterol...
4        Cardiology  A 61-year-old woman has persistent hypertensio...


In [7]:
# ---------------------------------------
# 3. Function to process one transcription
# ---------------------------------------

def extract_medical_data(transcription, medical_specialty):
    """
    Extract age, treatment/procedure and ICD code
    from a medical transcription using one OpenAI call.
    """

    messages = [
        {
            "role": "system",
            "content": """
You are a healthcare data extraction assistant.

Extract structured information from the medical transcription.

Return:
1. Patient age
2. Recommended treatment or procedure
3. ICD-10-CM code related to the recommended treatment,
   procedure, or medical condition.

If information is missing, return "Unknown".

The ICD code should be treated as an AI-generated suggestion
and should be verified by a qualified medical coding professional.
"""
        },
        {
            "role": "user",
            "content": f"""
Medical Specialty:
{medical_specialty}

Medical Transcription:
{transcription}

Extract the requested medical information.
"""
        }
    ]

    # ---------------------------------------
    # Function definition
    # ---------------------------------------

    tools = [
        {
            "type": "function",
            "function": {
                "name": "extract_medical_data",
                "description": "Extract structured medical information",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "Age": {
                            "type": "integer",
                            "description": "Patient's age"
                        },
                        "Recommended Treatment/Procedure": {
                            "type": "string",
                            "description": (
                                "Recommended treatment or medical procedure"
                            )
                        },
                        "ICD Code": {
                            "type": "string",
                            "description": (
                                "Suggested ICD-10-CM code related "
                                "to the condition, treatment, or procedure"
                            )
                        }
                    },
                    "required": [
                        "Age",
                        "Recommended Treatment/Procedure",
                        "ICD Code"
                    ]
                }
            }
        }
    ]

    # ---------------------------------------
    # ONE OpenAI API CALL
    # ---------------------------------------

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=tools,
        tool_choice={
            "type": "function",
            "function": {
                "name": "extract_medical_data"
            }
        }
    )

    # ---------------------------------------
    # Get function arguments
    # ---------------------------------------

    tool_call = response.choices[0].message.tool_calls[0]

    arguments = tool_call.function.arguments

    return json.loads(arguments)


In [8]:
# ---------------------------------------
# 4. Process the dataset
# ---------------------------------------

processed_data = []

for index, row in df.iterrows():

    print(
        f"Processing record {index + 1}/{len(df)}..."
    )

    try:

        medical_specialty = row["medical_specialty"]
        transcription = row["transcription"]

        # One OpenAI call
        extracted_data = extract_medical_data(
            transcription,
            medical_specialty
        )

        # Add medical specialty
        extracted_data["Medical Specialty"] = medical_specialty

        # Add original transcription
        extracted_data["Transcription"] = transcription

        # Store result
        processed_data.append(extracted_data)

        print(extracted_data)

    except Exception as e:

        print(
            f"Error processing row {index}: {e}"
        )



Processing record 1/54...
{'Age': 65, 'Recommended Treatment/Procedure': 'stress test and further cardiac evaluation', 'ICD Code': 'I20.9', 'Medical Specialty': 'Cardiology', 'Transcription': 'A 65-year-old male presents with chest pain on exertion. The cardiologist recommends a stress test and further cardiac evaluation.'}
Processing record 2/54...
{'Age': 58, 'Recommended Treatment/Procedure': 'ECG and 24-hour Holter monitoring', 'ICD Code': 'R00.2', 'Medical Specialty': 'Cardiology', 'Transcription': 'A 58-year-old woman reports intermittent palpitations. The physician recommends an ECG and 24-hour Holter monitoring.'}
Processing record 3/54...
{'Age': 72, 'Recommended Treatment/Procedure': 'echocardiogram and medication adjustment', 'ICD Code': 'I50.9', 'Medical Specialty': 'Cardiology', 'Transcription': 'A 72-year-old male has shortness of breath and swelling in both legs. The doctor recommends an echocardiogram and medication adjustment.'}
Processing record 4/54...
{'Age': 49, 'R

In [9]:
# ---------------------------------------
# 5. Create structured DataFrame
# ---------------------------------------

df_structured = pd.DataFrame(processed_data)

In [ ]:
# ---------------------------------------
# 6. Display final result
# ---------------------------------------

print("\n")
print("=" * 80)
print("FINAL STRUCTURED DATA")
print("=" * 80)

print(df_structured.to_string(index=False))

In [12]:
# ---------------------------------------
# 7. Save results
# ---------------------------------------

output_file = "structured_medical_data.csv"

df_structured.to_csv(
    output_file,
    index=False
)

print("\n")
print(f"Results saved to: {output_file}")



Results saved to: structured_medical_data.csv
